In [20]:
llmcfg_tests = []

In [19]:
agora_tests = []

In [18]:
llmd_tests =  []

## Step 2: 提取path

In [17]:
import json
from collections import defaultdict

def generate_test_paths_with_conditions(cfg_input):
    """
    生成所有携带条件信息的测试路径。
    :param cfg_input: cfg dict或者json字符串
    :return: test paths: 如 [['S1', 'S2', 'S3', "condition: Actor uses 'change' feature", 'S3a'], ...]
    """
    if isinstance(cfg_input, str):
        cfg = json.loads(cfg_input)
    elif isinstance(cfg_input, dict):
        cfg = cfg_input
    else:
        raise TypeError("输入必须是JSON字符串或字典")
    nodes = cfg["nodes"]
    edges = cfg["edges"]
    root_node = nodes[0]["id"]
    # 构建邻接表，边附带condition
    graph = defaultdict(list)
    for edge in edges:
        condition = edge.get("condition")
        graph[edge['from']].append((edge['to'], condition))

    all_paths = []

    def dfs(curr, path):
        if curr in path:  # 检查环路，剪枝
            all_paths.append(path.copy())
            return
        path = path + [curr]
        if curr not in graph or not graph[curr]:  # 终止节点
            all_paths.append(path)
            return
        for neighbor, condition in graph[curr]:
            path_with_condition = list(path)
            if condition:
                path_with_condition.append(f"condition: {condition}")
            dfs(neighbor, path_with_condition)

    dfs(root_node, [])
    return all_paths

def generate_test_scenarios_with_conditions(cfg_input):
    """
    将测试路径转为步进式的测试场景（含条件描述）。
    :param cfg_input: cfg dict或者json字符串
    :return: test scenarios: [[step1, step2, ..., stepN], ...]
    """
    test_paths = generate_test_paths_with_conditions(cfg_input)
    if isinstance(cfg_input, str):
        cfg_data = json.loads(cfg_input)
    else:
        cfg_data = cfg_input
    id_to_statement = {node["id"]: node["Statement"] for node in cfg_data["nodes"]}

    test_cases = []
    for path in test_paths:
        steps = []
        for item in path:
            if isinstance(item, str) and item.startswith("condition: "):
                # condition作为一个描述步骤
                steps.append(f"{item.split('condition: ')[1]}")
            else:
                steps.append(id_to_statement.get(item, f"Unknown node: {item}"))
        test_cases.append(steps)
    return test_cases

# ---------- 运行演示 ----------
if __name__ == '__main__':
    print("所有测试路径:")
    test_paths = generate_test_paths_with_conditions(GROUNDTRUTH_CFG)
    print(len(test_paths))
    for p in test_paths:
        print(p)
    print("\n所有测试场景步骤:")
    test_scenarios = generate_test_scenarios_with_conditions(GROUNDTRUTH_CFG)
    print(test_scenarios)
    # for i, scenario in enumerate(test_scenarios, 1):
    #     print(f"\nScenario {i}:")
        
        # for step in scenario:
        #     print(step)

所有测试路径:
4
['S1', 'S2', 'S3', 'S4', "condition: User selects the sub-option 'Add a measurement'", 'S3a']
['S1', 'S2', 'S3', 'S4', "condition: User selects 'display trend'", 'S3b']
['S1', 'S2', 'S3', 'condition: There is an error in displaying the measurements', 'E2']
['S1', 'S2', 'condition: No measurements found', 'E1']

所有测试场景步骤:
[["The user selects the 'View Vitals Measurements' tab for the given patient", 'The system searches for the measurements', 'The system displays them in a table format, including the date and time of each measurement and the corresponding vital sign values', 'The user views the results', "User selects the sub-option 'Add a measurement'", 'The system starts the use case CPM:US1.002– Add a measurement'], ["The user selects the 'View Vitals Measurements' tab for the given patient", 'The system searches for the measurements', 'The system displays them in a table format, including the date and time of each measurement and the corresponding vital sign values', 'The 

## Step3: run_assessment.py 最终版本

In [22]:
import json
import re
import time
import requests
import pandas as pd
from typing import List, Dict, Any, Optional
from tqdm.notebook import tqdm

# ==========================================
# 1. 核心配置
# ==========================================
LLM_API_URL = "http://xxxxx/api/chat/completions" # 替换为你的 URL
LLM_API_KEY = "sk-xxxxxxx" # 替换为你的 Key 

# ==========================================
# 2. ACE 评估器类 (Version 2.0 - Academic Ready)
# ==========================================
class ACEEvaluator:
    def __init__(self, api_url: str, api_key: str):
        self.url = api_url
        self.key = api_key

    def _extract_json(self, text: str) -> Optional[Dict]:
        """从 LLM 回复中稳健提取 JSON 数据"""
        if not text: return None
        match = re.search(r'```json\n?(.*?)\n?```', text, re.DOTALL)
        content = match.group(1) if match else text
        try:
            return json.loads(content.strip())
        except:
            match_fallback = re.search(r'\{.*\}', content, re.DOTALL)
            if match_fallback:
                try: return json.loads(match_fallback.group())
                except: pass
        return None

    def call_mapping_llm(self, test_case: Dict, all_paths: List[List[str]]) -> Dict:
        """调用 LLM 进行语义映射与术语一致性评估"""
        paths_txt = ""
        for i, path in enumerate(all_paths):
            nodes = " -> ".join([f"[{j}: {node}]" for j, node in enumerate(path)])
            paths_txt += f"Path_{i}: {nodes}\n"

        mapping_prompt = f"""
# Task: Map Test Case to Ground Truth Path & Evaluate Quality
# Paths (Ground Truth):
{paths_txt}

# Test Case (Generated):
{json.dumps(test_case, indent=2)}

# Rules:
1. Intent Matching: Find the most relevant 'matched_path_id' (e.g., "Path_0"). If no logical match, set to null.
2. Indexing: List the 'covered_node_indices' as integers.
3. Terminology Consistency: Rate 'term_consistency' (1-5) between TC steps and Path nodes (5=Identical terms).
4. Logic Status: Set 'status' to "FULL" if intent is complete, else "PARTIAL".

# Output strictly JSON:
{{
  "matched_path_id": "Path_0",
  "confidence": 0.95,
  "covered_node_indices": [0, 1, 3],
  "term_consistency": 5,
  "status": "FULL"
}}
"""
        data = {
            # "modelType": "claude-opus-4-6",
            # "modelType": "gpt-5.4",
            # "modelType": "gpt-5.2",
            "modelType": "gpt-5.4-nano",
            # "modelType": "claude-opus-4-6",
            # "modelType": "gemini-3-flash",
            # "modelType": "qwen3.5-plus",
            "message": mapping_prompt,
            "temperature": 0.0,      # 严禁随机性，确保评估一致性
            "top_p": 1,
            "max_tokens": 1000
        }
        headers = {"Content-Type": "application/json", "Authorization": f"Bearer {self.key}"}

        try:
            response = requests.post(self.url, headers=headers, json=data, verify=False, timeout=60)
            response.raise_for_status()
            res_content = response.json()["data"]["chat"]
            parsed = self._extract_json(res_content)
            return parsed if parsed else {}
        except Exception as e:
            print(f"Error mapping TC {test_case.get('ID')}: {e}")
            return {"matched_path_id": None, "confidence": 0, "covered_node_indices": [], "term_consistency": 0}

    def evaluate(self,
                 test_cases: List[Dict],
                 ground_truth_paths: List[List[str]],
                 method_name="Method",
                 path_threshold=0.8,
                 min_coverage_ratio=0.2):

        path_coverage = {f"Path_{i}": set() for i in range(len(ground_truth_paths))}
        path_contributors = {f"Path_{i}": [] for i in range(len(ground_truth_paths))}
        path_lengths = {f"Path_{i}": len(p) for i, p in enumerate(ground_truth_paths)}

        results = []
        invalid_count = 0
        order_error_count = 0
        redundant_count = 0

        # ======================
        # Mapping + Filtering
        # ======================
        for tc in tqdm(test_cases, desc=f"Evaluating {method_name}"):
            mapping = self.call_mapping_llm(tc, ground_truth_paths)

            pid_raw = mapping.get("matched_path_id")
            match = re.search(r'path[_\s]?(\d+)', str(pid_raw), re.I)
            if not match:
                invalid_count += 1
                continue

            pid = f"Path_{int(match.group(1))}"
            if pid not in path_lengths:
                invalid_count += 1
                continue

            # ---------- indices ----------
            raw_indices = [int(i) for i in mapping.get("covered_node_indices", [])]
            path_len = path_lengths[pid]
            valid_indices = [i for i in raw_indices if 0 <= i < path_len]

            if not valid_indices:
                invalid_count += 1
                continue

            # ---------- coverage ----------
            unique_indices = sorted(set(valid_indices))
            coverage_ratio = len(unique_indices) / path_len if path_len > 0 else 0
            has_last = (path_len - 1) in unique_indices

            # ---------- order ----------
            clean_indices = list(dict.fromkeys(valid_indices))
            is_monotonic = all(clean_indices[i] <= clean_indices[i+1]
                               for i in range(len(clean_indices)-1))

            if not is_monotonic:
                order_error_count += 1

            # ---------- validity (核心) ----------
            is_valid = (
                (coverage_ratio >= min_coverage_ratio) or 
                (has_last and len(unique_indices) >= 1 and confidence >= 0.5)
            )

            if not is_valid:
                invalid_count += 1
                continue

            # ---------- record ----------
            results.append(mapping)
            path_coverage[pid].update(unique_indices)
            tc_consistency = float(mapping.get("term_consistency", 0))

            path_contributors[pid].append({
                "id": tc.get("ID", "unknown"),
                "nodes": set(unique_indices)
            })

        # ======================
        # Redundancy
        # ======================
        effective_tc_count = len(results)
        non_redundant_count = 0

        for pid, contributors in path_contributors.items():
            contributors.sort(key=lambda x: len(x["nodes"]), reverse=True)
            removed = set()

            for i in range(len(contributors)):
                if i in removed:
                    continue
                for j in range(i + 1, len(contributors)):
                    if j in removed:
                        continue
                    if contributors[j]["nodes"].issubset(contributors[i]["nodes"]):
                        redundant_count += 1
                        removed.add(j)

            non_redundant_count += len(contributors) - len(removed)

        # ======================
        # Coverage Metrics
        # ======================
        total_nodes = sum(path_lengths.values())
        covered_nodes = sum(len(v) for v in path_coverage.values())
        step_cov = covered_nodes / total_nodes if total_nodes else 0

        covered_paths = 0

        for pid, idxs in path_coverage.items():
            path_len = path_lengths[pid]
            if path_len == 0:
                continue

            ratio = len(idxs) / path_len
            has_last = (path_len - 1) in idxs

            if ratio >= path_threshold and has_last:
                covered_paths += 1

        pcr = covered_paths / len(ground_truth_paths) if ground_truth_paths else 0
        acpr = non_redundant_count / covered_paths if covered_paths else 0

        noise = invalid_count / len(test_cases) if test_cases else 0
        order_err = order_error_count / effective_tc_count if effective_tc_count else 0
        redundancy = redundant_count / effective_tc_count if effective_tc_count else 0
        effective_tc_count = len(results)
        # 1. 统计所有有效 TC 的总得分
        total_consistency = sum(float(r.get('term_consistency', 0)) for r in results)
        
        # 2. 直接算简单平均（针对有效用例）
        avg_consistency = total_consistency / len(results) if results else 0



        print("\n" + "="*70)
        print(f"{method_name}")
        print("="*70)
        print(f"Step Coverage: {step_cov:.2%}")
        print(f"Path Coverage: {pcr:.2%}")
        print(f"Noise: {noise:.2%}")
        print(f"Order Error: {order_err:.2%}")
        print(f"Redundancy: {redundancy:.2%}")
        print(f"ACPR: {acpr:.2f}")
        print(f"Valid TC: {effective_tc_count}/{len(test_cases)}")
        print(f"Avg Consistency: {avg_consistency:.2f}/5.0")

        return {
            "Step_Cov": step_cov,
            "PCR": pcr,
            "Noise": noise,
            "Order_Error": order_err,
            "Redundancy": redundancy,
            "ACPR": acpr,
            "Consistency": avg_consistency  # 补上这个
        }


# ==========================================
# 3. 结果汇总对比函数
# ==========================================
def compare_results(results: List[Dict], names: List[str]):
    df = pd.DataFrame(results)
    df.insert(0, "Method", names)

    # 格式化百分比
    for col in ["Step_Cov", "PCR", "Noise", "Redundancy"]:
        df[col] = df[col].map(lambda x: f"{x:.2%}")

    # 格式化数值
    df["ACPR"] = df["ACPR"].map(lambda x: f"{x:.2f}")
    if "Consistency" in df.columns:
        df["Consistency"] = df["Consistency"].map(lambda x: f"{x:.2f}")

    return df

# ==========================================
# 使用示例 (请替换为你自己的数据)
# ==========================================
print("所有测试路径:")
test_paths = generate_test_paths_with_conditions(GROUNDTRUTH_CFG)
print(len(test_paths))
for p in test_paths:
    print(p)
print("\n所有测试场景步骤:")
ground_truth_paths = generate_test_scenarios_with_conditions(GROUNDTRUTH_CFG)

# print("ground_truth_paths:", ground_truth_paths)

evaluator = ACEEvaluator(LLM_API_URL, LLM_API_KEY)
report_ours = evaluator.evaluate(llmcfg_tests, ground_truth_paths, method_name="LLMCFG-TGen")
report_agora = evaluator.evaluate(agora_tests, ground_truth_paths, method_name="Agora")
report_llmd = evaluator.evaluate(llmd_tests, ground_truth_paths, method_name="LLMD")
final_table = compare_results([report_ours, report_agora, report_llmd], ["LLMCFG-TGen", "Agora", "LLMD"])
display(final_table)

所有测试路径:
4
['S1', 'S2', 'S3', 'S4', "condition: User selects the sub-option 'Add a measurement'", 'S3a']
['S1', 'S2', 'S3', 'S4', "condition: User selects 'display trend'", 'S3b']
['S1', 'S2', 'S3', 'condition: There is an error in displaying the measurements', 'E2']
['S1', 'S2', 'condition: No measurements found', 'E1']

所有测试场景步骤:


Evaluating LLMCFG-TGen:   0%|          | 0/4 [00:00<?, ?it/s]


LLMCFG-TGen
Step Coverage: 85.71%
Path Coverage: 25.00%
Noise: 0.00%
Order Error: 0.00%
Redundancy: 0.00%
ACPR: 4.00
Valid TC: 4/4
Avg Consistency: 4.00/5.0


Evaluating Agora:   0%|          | 0/3 [00:00<?, ?it/s]


Agora
Step Coverage: 52.38%
Path Coverage: 0.00%
Noise: 0.00%
Order Error: 0.00%
Redundancy: 0.00%
ACPR: 0.00
Valid TC: 3/3
Avg Consistency: 4.67/5.0


Evaluating LLMD:   0%|          | 0/10 [00:00<?, ?it/s]


LLMD
Step Coverage: 76.19%
Path Coverage: 0.00%
Noise: 0.00%
Order Error: 0.00%
Redundancy: 40.00%
ACPR: 0.00
Valid TC: 10/10
Avg Consistency: 4.30/5.0


,Method,Step_Cov,PCR,Noise,Order_Error,Redundancy,ACPR,Consistency
0,LLMCFG-TGen,85.71%,25.00%,0.00%,0.0,0.00%,4.00,4.00
1,Agora,52.38%,0.00%,0.00%,0.0,0.00%,0.00,4.67
2,LLMD,76.19%,0.00%,0.00%,0.0,40.00%,0.00,4.30
